# 03.05 · Normalization, regularization, initialization

**Chapter:** Deep Learning · **Milestone:** M1

## Goals
- [ ] Debug training: overfit one batch first; LR range test
- [ ] See why initialization matters (activation statistics through depth)
- [ ] Residual connections + LayerNorm: why deep nets (and transformers) train at all
- [ ] Dropout, weight decay, gradient clipping

## Resources
- [Karpathy – A Recipe for Training Neural Networks](https://karpathy.github.io/2019/04/25/recipe/) — free
- [Karpathy – Building makemore Part 3: Activations, gradients, BatchNorm](https://www.youtube.com/watch?v=P6sfmUTpUmc) — free
- [Google – Deep Learning Tuning Playbook](https://github.com/google-research/tuning_playbook) — free

The transformer block you'll build in 04.03 is: `x = x + Attention(LayerNorm(x)); x = x + MLP(LayerNorm(x))`.
This lesson explains every piece except attention.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from aieng import DATA

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.286,), (0.353,))])
train_set = Subset(datasets.FashionMNIST(DATA, train=True, download=True, transform=tfm), range(10_000))
val_set = Subset(datasets.FashionMNIST(DATA, train=False, download=True, transform=tfm), range(2_000))
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = DataLoader(val_set, batch_size=1000)


def evaluate(model):
    model.eval()
    with torch.no_grad():
        return sum((model(x.to(device)).argmax(1).cpu() == y).sum().item() for x, y in val_loader) / len(val_set)

## 1. Rule #1: overfit a single batch
Before a long run, check the model can drive the loss to ~0 on one batch. If it can't, you have a bug
(wrong labels, wrong loss, broken data pipeline) — no hyperparameter will save you.

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.GELU(), nn.Linear(256, 10)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
xb, yb = next(iter(train_loader))
xb, yb = xb.to(device), yb.to(device)
for step in range(200):
    loss = nn.functional.cross_entropy(model(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
print(f"loss on one batch after 200 steps: {loss.item():.4f}  (≈0 means the pipeline works)")

## 2. Learning-rate range test
Increase the LR exponentially each step; plot loss. Pick an LR a bit below where loss is lowest/falling fastest.

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.GELU(), nn.Linear(256, 10)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-6)
lrs, losses = [], []
lr = 1e-6
for xb, yb in train_loader:
    for g in opt.param_groups: g["lr"] = lr
    loss = nn.functional.cross_entropy(model(xb.to(device)), yb.to(device))
    opt.zero_grad(); loss.backward(); opt.step()
    lrs.append(lr); losses.append(loss.item())
    lr *= 1.2
    if loss.item() > 10 or lr > 10: break
plt.semilogx(lrs, losses); plt.xlabel("learning rate"); plt.ylabel("loss"); plt.ylim(0, 5); plt.show()

## 3. Initialization: keep activations healthy
With bad init, activations shrink to 0 or explode as they pass through layers, and gradients do the same.
Below: a 20-layer tanh net; we print the std of activations at each layer for three init scales.

In [ ]:
def activation_stds(scale, depth=20, width=256):
    torch.manual_seed(0)
    x = torch.randn(512, width)
    stds = []
    for _ in range(depth):
        W = torch.randn(width, width) * scale / width**0.5
        x = torch.tanh(x @ W)
        stds.append(x.std().item())
    return stds

for scale, label in [(0.5, "too small"), (1.0, "~Xavier (1/√n)"), (3.0, "too large")]:
    plt.plot(activation_stds(scale), label=f"{label}")
plt.xlabel("layer"); plt.ylabel("activation std"); plt.legend(); plt.show()

Too small → signal dies. Too large → tanh saturates at ±1 (gradient ≈ 0). PyTorch's defaults (Kaiming/Xavier-style) handle
this for standard layers; LLM code often uses small normal init (std ≈ 0.02) plus normalization layers.

## 4. Residual connections + LayerNorm
- **Residual**: `x = x + f(x)`. Gradients flow straight through the `+`, so very deep nets train.
- **LayerNorm**: normalise each example's features to mean 0 / std 1 (then learnable scale & shift). Unlike BatchNorm it doesn't
  depend on the batch, so it works for sequences and batch size 1 — that's why transformers use it (or its cousin RMSNorm).

Experiment: a 20-layer plain MLP vs the same depth with residuals + LayerNorm.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, residual, norm):
        super().__init__()
        self.norm = nn.LayerNorm(d) if norm else nn.Identity()
        self.ff = nn.Sequential(nn.Linear(d, d), nn.GELU())
        self.residual = residual

    def forward(self, x):
        out = self.ff(self.norm(x))
        return x + out if self.residual else out


def deep_net(residual, norm, depth=20, d=128):
    return nn.Sequential(nn.Flatten(), nn.Linear(784, d), *[Block(d, residual, norm) for _ in range(depth)], nn.Linear(d, 10)).to(device)


def train(model, epochs=3, lr=1e-3, wd=0.0, clip=None):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            loss = nn.functional.cross_entropy(model(xb.to(device)), yb.to(device))
            opt.zero_grad(); loss.backward()
            if clip:
                nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
    return evaluate(model)


for residual, norm in [(False, False), (True, False), (True, True)]:
    torch.manual_seed(0)
    print(f"20 layers | residual={residual!s:5} layernorm={norm!s:5} -> val acc {train(deep_net(residual, norm)):.3f}")

Typical result: the plain 20-layer net is stuck near chance (10%); residuals alone fix most of it. LayerNorm's benefit is small here
but grows with higher learning rates, more depth and longer runs (try lr=3e-3 or depth 50) — that's the regime LLMs live in.

## 5. Regularization: dropout & weight decay
Small training set (10k) + big model = overfitting. **Dropout** randomly zeroes activations during training (`model.train()`),
and is off at inference (`model.eval()` — forgetting this is a classic bug). **Weight decay** (AdamW) pulls weights toward 0.

In [ ]:
def wide(p_drop):
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 1024), nn.GELU(), nn.Dropout(p_drop),
                         nn.Linear(1024, 1024), nn.GELU(), nn.Dropout(p_drop), nn.Linear(1024, 10)).to(device)

for p_drop, wd in [(0.0, 0.0), (0.3, 0.0), (0.3, 0.1)]:
    torch.manual_seed(0)
    print(f"dropout={p_drop} weight_decay={wd} -> val acc {train(wide(p_drop), epochs=5, wd=wd):.3f}")

## 6. Gradient clipping
Rescale gradients when their total norm exceeds a threshold. Cheap insurance against loss spikes; standard in LLM training
(`clip_grad_norm_(params, 1.0)`). The `train` function above supports `clip=`.

## Exercises
1. Break a run three ways — LR ×100, init std ×10, remove LayerNorm from the 20-layer residual net — and describe each loss curve.
2. Replace `nn.LayerNorm` with an RMSNorm you write yourself (`x / sqrt(mean(x²) + eps) * g`). Same accuracy?
3. Plot the gradient norm per layer for the plain vs residual 20-layer nets after one backward pass.

In [ ]:
# your code here